In [1]:
from pathlib import Path
import joblib
import pandas as pd
import numpy as np

model_path = Path("models/market_segmentation_bundle.joblib")

bundle = joblib.load(model_path)

scaler = bundle["scaler"]
kmeans = bundle["kmeans"]
feature_cols = bundle["feature_cols"]
cluster_labels = bundle["cluster_labels"]

print("Model loaded successfully.")

Model loaded successfully.


In [2]:
print("Training year:", bundle["training_year"])
print("HS code:", bundle["hs_code"])
print("Features:")
for i, col in enumerate(feature_cols, start=1):
    print(i, col)

print("\nNumber of clusters:", kmeans.n_clusters)
print("Scaler features:", scaler.n_features_in_)

Training year: 2024
HS code: 850760
Features:
1 log_export_2024
2 growth_symmetric_2024
3 momentum_symmetric_2024
4 monthly_cv_2024
5 active_month_ratio_2024

Number of clusters: 6
Scaler features: 5


In [3]:
baseline_2024 = pd.read_csv(
    "outputs/market_opportunity_ranking_2024.csv"
)

baseline_2024[
    [
        "opportunity_rank",
        "destination",
        "cluster",
        "market_segment",
        "opportunity_score"
    ]
].head(10)

,opportunity_rank,destination,cluster,market_segment,opportunity_score
0,1,France,1,成熟核心市场,81.314696
1,2,Australia,1,成熟核心市场,80.429729
2,3,Uzbekistan,0,高增长机会市场,78.001788
3,4,Romania,1,成熟核心市场,77.755891
4,5,Ukraine,1,成熟核心市场,77.181517
5,6,Saudi Arabia,1,成熟核心市场,75.225910
6,7,USA,1,成熟核心市场,74.878498
7,8,Hungary,1,成熟核心市场,74.780928
8,9,United Arab Emirates,1,成熟核心市场,74.189985
9,10,Serbia,0,高增长机会市场,73.810751


In [4]:
import comtradeapicall

TARGET_YEAR = 2025
REPORTER_CODE = "156"       # China
HS_CODE = "850760"
FLOW_CODE = "X"             # Export

periods_2025 = ",".join(
    f"{TARGET_YEAR}{month:02d}"
    for month in range(1, 13)
)

periods_2025

'202501,202502,202503,202504,202505,202506,202507,202508,202509,202510,202511,202512'

In [5]:
import comtradeapicall

TARGET_YEAR = 2025
REPORTER_CODE = "156"       # China
HS_CODE = "850760"
FLOW_CODE = "X"             # Export

periods_2025 = ",".join(
    f"{TARGET_YEAR}{month:02d}"
    for month in range(1, 13)
)

periods_2025

'202501,202502,202503,202504,202505,202506,202507,202508,202509,202510,202511,202512'

## 没有25年数据

In [6]:
test_2022 = comtradeapicall.previewFinalData(
    typeCode="C",
    freqCode="M",
    clCode="HS",
    period="202201",
    reporterCode="156",
    cmdCode="850760",
    flowCode="X",
    partnerCode=None,
    partner2Code=None,
    customsCode=None,
    motCode=None,
    maxRecords=500,
    format_output="JSON",
    aggregateBy=None,
    breakdownMode="classic",
    countOnly=None,
    includeDesc=True
)

print("shape:", test_2022.shape)
print("columns:", test_2022.columns.tolist())

display(test_2022.head())

shape: (163, 47)
columns: ['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period', 'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc', 'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code', 'partner2ISO', 'partner2Desc', 'classificationCode', 'classificationSearchCode', 'isOriginalClassification', 'cmdCode', 'cmdDesc', 'aggrLevel', 'isLeaf', 'customsCode', 'customsDesc', 'mosCode', 'motCode', 'motDesc', 'qtyUnitCode', 'qtyUnitAbbr', 'qty', 'isQtyEstimated', 'altQtyUnitCode', 'altQtyUnitAbbr', 'altQty', 'isAltQtyEstimated', 'netWgt', 'isNetWgtEstimated', 'grossWgt', 'isGrossWgtEstimated', 'cifvalue', 'fobvalue', 'primaryValue', 'legacyEstimationFlag', 'isReported', 'isAggregate']


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,M,20220101,2022,1,202201,156,CHN,China,X,...,110733776.0,False,0.0,False,None,3.319252e+09,3.319252e+09,0,False,True
1,C,M,20220101,2022,1,202201,156,CHN,China,X,...,3035.0,False,0.0,False,None,1.162700e+04,1.162700e+04,0,True,False
2,C,M,20220101,2022,1,202201,156,CHN,China,X,...,3243.0,False,0.0,False,None,7.786700e+04,7.786700e+04,0,True,False
3,C,M,20220101,2022,1,202201,156,CHN,China,X,...,2140.0,False,0.0,False,None,6.657500e+04,6.657500e+04,0,True,False
4,C,M,20220101,2022,1,202201,156,CHN,China,X,...,4935.0,False,0.0,False,None,9.484700e+04,9.484700e+04,0,True,False


In [7]:
periods_2022 = ",".join(
    f"2022{month:02d}"
    for month in range(1, 13)
)

periods_2022

'202201,202202,202203,202204,202205,202206,202207,202208,202209,202210,202211,202212'

In [8]:
raw_2022 = comtradeapicall._previewFinalData(
    typeCode="C",
    freqCode="M",
    clCode="HS",
    period=periods_2022,
    reporterCode="156",
    cmdCode="850760",
    flowCode="X",
    partnerCode=None,
    partner2Code=None,
    customsCode=None,
    motCode=None,
    maxRecords=500,
    format_output="JSON",
    aggregateBy=None,
    breakdownMode="classic",
    countOnly=None,
    includeDesc=True
)

print("shape:", raw_2022.shape)
print("months:", raw_2022["period"].nunique())

{ "statusCode": 429, "message": "Rate limit is exceeded. Try again in 2 seconds." }
{ "statusCode": 429, "message": "Rate limit is exceeded. Try again in 2 seconds." }
{ "statusCode": 429, "message": "Rate limit is exceeded. Try again in 2 seconds." }
shape: (1482, 47)
months: 9


In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

# --------------------------------------------------
# Project paths
# --------------------------------------------------

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


# --------------------------------------------------
# 1. 清洗刚刚拉取的 2022 原始数据
#    完全沿用 01 的口径
# --------------------------------------------------

partner_monthly_2022 = (
    raw_2022[
        raw_2022["isAggregate"] == False
    ][
        [
            "period",
            "partnerCode",
            "partnerISO",
            "partnerDesc",
            "primaryValue"
        ]
    ]
    .copy()
)

partner_monthly_2022 = partner_monthly_2022.rename(
    columns={
        "partnerCode": "destination_code",
        "partnerISO": "destination_iso",
        "partnerDesc": "destination",
        "primaryValue": "export_value_usd"
    }
)

partner_monthly_2022["month"] = pd.to_datetime(
    partner_monthly_2022["period"].astype(str),
    format="%Y%m"
)

partner_monthly_2022["year"] = (
    partner_monthly_2022["month"].dt.year
)

In [10]:
partner_monthly_2023_2024 = pd.read_csv(
    PROCESSED_DIR
    / "china_hs850760_partner_monthly_2023_2024.csv"
)

partner_monthly_2023_2024["month"] = pd.to_datetime(
    partner_monthly_2023_2024["month"]
)

partner_monthly_2023_2024["year"] = (
    partner_monthly_2023_2024["month"].dt.year
)

In [11]:
partner_monthly_all = pd.concat(
    [
        partner_monthly_2022,
        partner_monthly_2023_2024
    ],
    ignore_index=True
)

print("shape:", partner_monthly_all.shape)

print(
    "years:",
    sorted(partner_monthly_all["year"].unique())
)

print(
    "duplicate market-month:",
    partner_monthly_all.duplicated(
        subset=["destination_code", "month"]
    ).sum()
)

shape: (5667, 7)
years: [np.int32(2022), np.int32(2023), np.int32(2024)]
duplicate market-month: 0


In [12]:
def build_market_features(monthly_data, target_year):
    """
    根据 target_year-1 和 target_year 的月度出口数据，
    构造与 02_market_segmentation_ml 完全同口径的市场特征。
    """

    previous_year = target_year - 1

    data = monthly_data.copy()

    # --------------------------------------------------
    # 1. 只保留前一年 + 目标年
    # --------------------------------------------------

    data = data[
        data["year"].isin(
            [previous_year, target_year]
        )
    ].copy()


    # --------------------------------------------------
    # 2. 构建完整 Market × Month 面板
    #    缺失月份出口额补 0
    # --------------------------------------------------

    all_markets = (
        data[
            ["destination_code", "destination"]
        ]
        .drop_duplicates()
    )

    all_months = pd.DataFrame(
        {
            "month": pd.date_range(
                start=f"{previous_year}-01-01",
                end=f"{target_year}-12-01",
                freq="MS"
            )
        }
    )

    full_panel = all_markets.merge(
        all_months,
        how="cross"
    )

    monthly_values = data[
        [
            "destination_code",
            "month",
            "export_value_usd"
        ]
    ].copy()

    full_panel = full_panel.merge(
        monthly_values,
        on=["destination_code", "month"],
        how="left"
    )

    full_panel["export_value_usd"] = (
        full_panel["export_value_usd"]
        .fillna(0)
    )

    full_panel["year"] = (
        full_panel["month"].dt.year
    )

    full_panel["month_num"] = (
        full_panel["month"].dt.month
    )


    # --------------------------------------------------
    # 3. 年度出口额：previous_year → target_year
    # --------------------------------------------------

    country_year = (
        full_panel
        .groupby(
            [
                "destination_code",
                "destination",
                "year"
            ],
            as_index=False
        )["export_value_usd"]
        .sum()
        .pivot(
            index=[
                "destination_code",
                "destination"
            ],
            columns="year",
            values="export_value_usd"
        )
        .reset_index()
    )

    country_year.columns.name = None

    country_year = country_year.rename(
        columns={
            previous_year: "export_previous",
            target_year: "export_target"
        }
    )


    # --------------------------------------------------
    # 4. 目标年份 12 个月数据
    # --------------------------------------------------

    panel_target = full_panel[
        full_panel["year"] == target_year
    ].copy()


    # --------------------------------------------------
    # 5. 月度稳定性 + 活跃度
    # --------------------------------------------------

    monthly_features = (
        panel_target
        .groupby(
            [
                "destination_code",
                "destination"
            ],
            as_index=False
        )
        .agg(
            export_target_from_months=(
                "export_value_usd",
                "sum"
            ),
            avg_monthly_export=(
                "export_value_usd",
                "mean"
            ),
            std_monthly_export=(
                "export_value_usd",
                "std"
            ),
            active_months=(
                "export_value_usd",
                lambda x: (x > 0).sum()
            )
        )
    )

    monthly_features["active_month_ratio"] = (
        monthly_features["active_months"] / 12
    )

    monthly_features["monthly_cv"] = (
        monthly_features["std_monthly_export"]
        /
        monthly_features["avg_monthly_export"]
    )


    # --------------------------------------------------
    # 6. H1 / H2 近期动量
    # --------------------------------------------------

    h1_h2 = (
        panel_target
        .assign(
            half=np.where(
                panel_target["month_num"] <= 6,
                "H1",
                "H2"
            )
        )
        .pivot_table(
            index=[
                "destination_code",
                "destination"
            ],
            columns="half",
            values="export_value_usd",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    h1_h2.columns.name = None


    # --------------------------------------------------
    # 7. 合并
    # --------------------------------------------------

    features = (
        monthly_features
        .merge(
            country_year[
                [
                    "destination_code",
                    "export_previous",
                    "export_target"
                ]
            ],
            on="destination_code",
            how="left"
        )
        .merge(
            h1_h2[
                [
                    "destination_code",
                    "H1",
                    "H2"
                ]
            ],
            on="destination_code",
            how="left"
        )
    )


    # --------------------------------------------------
    # 8. 原始业务指标
    # --------------------------------------------------

    features["growth"] = np.where(
        features["export_previous"] > 0,
        (
            features["export_target"]
            - features["export_previous"]
        )
        / features["export_previous"],
        np.nan
    )

    features["h2_h1_momentum"] = np.where(
        features["H1"] > 0,
        (
            features["H2"]
            - features["H1"]
        )
        / features["H1"],
        np.nan
    )


    # --------------------------------------------------
    # 9. 五个最终模型特征
    # --------------------------------------------------

    # 市场规模
    features["log_export"] = np.log1p(
        features["export_target"]
    )

    # 对称年度增长率
    features["growth_symmetric"] = np.where(
        (
            features["export_previous"]
            + features["export_target"]
        ) > 0,

        2 * (
            features["export_target"]
            - features["export_previous"]
        )
        / (
            features["export_target"]
            + features["export_previous"]
        ),

        0
    )

    # 对称半年动量
    features["momentum_symmetric"] = np.where(
        (features["H1"] + features["H2"]) > 0,

        2 * (
            features["H2"]
            - features["H1"]
        )
        / (
            features["H2"]
            + features["H1"]
        ),

        0
    )


    # --------------------------------------------------
    # 10. 与 02 完全相同的建模市场过滤
    # --------------------------------------------------

    is_non_market = (
        features["destination"]
        .str.contains(
            r"\bnes\b",
            case=False,
            na=False,
            regex=True
        )
    )

    features = features[
        (~is_non_market)
        &
        (features["export_target"] > 0)
    ].copy()


    # --------------------------------------------------
    # 11. 返回最终结果
    # --------------------------------------------------

    return features.reset_index(drop=True)

In [13]:
features_2023 = build_market_features(
    partner_monthly_all,
    target_year=2023
)

features_2024 = build_market_features(
    partner_monthly_all,
    target_year=2024
)

print("2023 markets:", features_2023.shape)
print("2024 markets:", features_2024.shape)

2023 markets: (196, 17)
2024 markets: (197, 17)


In [14]:
model_feature_cols = [
    "log_export",
    "growth_symmetric",
    "momentum_symmetric",
    "monthly_cv",
    "active_month_ratio"
]

features_2023[
    [
        "destination",
        "export_previous",
        "export_target"
    ]
    + model_feature_cols
].head(10)

,destination,export_previous,export_target,log_export,growth_symmetric,momentum_symmetric,monthly_cv,active_month_ratio
0,Afghanistan,564230.0,2386507.0,14.685342,1.235133,1.494444,1.213923,0.666667
1,Albania,288474.0,719209.0,13.485909,0.854902,-0.132526,1.260711,1.000000
2,Algeria,3922674.0,18573541.0,16.737249,1.302518,-0.538084,1.506923,1.000000
3,Andorra,107702.0,67429.0,11.118845,-0.459919,0.379570,2.295519,0.500000
4,Angola,3955145.0,2519493.0,14.739569,-0.443469,1.655694,2.350480,1.000000
5,Antigua and Barbuda,210853.0,206344.0,12.237305,-0.021616,1.349572,2.366216,0.416667
6,Argentina,39537494.0,68939228.0,18.048736,0.542084,-0.918510,0.849339,1.000000
7,Australia,534582104.0,723825408.0,20.400061,0.300766,0.803437,1.070272,1.000000
8,Austria,110983357.0,395661215.0,19.796069,1.123777,0.583228,0.672741,1.000000
9,Bahamas,5614474.0,503003.0,13.128353,-1.671104,0.540573,1.110170,1.000000


In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 2023：训练数据
X_2023 = features_2023[
    model_feature_cols
].copy()

# 只在 2023 上学习均值和标准差
historical_scaler = StandardScaler()

X_2023_scaled = historical_scaler.fit_transform(
    X_2023
)

# 只在 2023 上训练 KMeans
historical_kmeans = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=20
)

features_2023["cluster_2023"] = (
    historical_kmeans.fit_predict(
        X_2023_scaled
    )
)

In [16]:
# 2024：真正的新数据
X_2024 = features_2024[
    model_feature_cols
].copy()

# 用 2023 学到的标准化规则
X_2024_scaled = historical_scaler.transform(
    X_2024
)

# 用 2023 学到的聚类中心判断 2024
features_2024["cluster_2024_pred"] = (
    historical_kmeans.predict(
        X_2024_scaled
    )
)

In [17]:
historical_cluster_profile = (
    features_2023
    .groupby("cluster_2023")
    .agg(
        market_count=(
            "destination",
            "count"
        ),
        avg_export=(
            "export_target",
            "mean"
        ),
        avg_growth=(
            "growth_symmetric",
            "mean"
        ),
        avg_momentum=(
            "momentum_symmetric",
            "mean"
        ),
        avg_cv=(
            "monthly_cv",
            "mean"
        ),
        avg_activity=(
            "active_month_ratio",
            "mean"
        )
    )
    .round(3)
)

historical_cluster_profile

,market_count,avg_export,avg_growth,avg_momentum,avg_cv,avg_activity
cluster_2023,,,,,,
0,45,7.620199e+06,0.993,0.844,1.358,0.919
1,51,7.820712e+06,0.022,-0.021,1.026,0.972
2,21,1.570144e+06,1.291,-1.450,2.331,0.516
3,10,1.149635e+06,1.599,1.848,2.755,0.342
4,59,1.078076e+09,0.410,0.138,0.443,1.000
5,10,2.637100e+05,-1.013,0.757,2.370,0.367


In [18]:
from sklearn.metrics import adjusted_rand_score

validation_2024 = (
    features_2024[
        [
            "destination",
            "cluster_2024_pred"
        ]
    ]
    .merge(
        baseline_2024[
            [
                "destination",
                "cluster"
            ]
        ],
        on="destination",
        how="inner"
    )
)

print("Compared markets:", len(validation_2024))

ari = adjusted_rand_score(
    validation_2024["cluster"],
    validation_2024["cluster_2024_pred"]
)

print("Adjusted Rand Index:", round(ari, 4))

Compared markets: 197
Adjusted Rand Index: 0.8316


In [19]:
pd.crosstab(
    validation_2024["cluster"],
    validation_2024["cluster_2024_pred"],
    rownames=["2024重新训练模型"],
    colnames=["2023旧模型预测2024"]
)

2023旧模型预测2024,0,1,2,3,4,5
2024重新训练模型,,,,,,
0,38,0,0,2,0,0
1,0,6,0,0,63,0
2,0,0,0,0,0,11
3,0,0,1,0,0,6
4,1,57,0,0,2,0
5,0,0,5,5,0,0


In [20]:
market_transition = (
    features_2023[
        [
            "destination",
            "cluster_2023"
        ]
    ]
    .merge(
        features_2024[
            [
                "destination",
                "cluster_2024_pred"
            ]
        ],
        on="destination",
        how="inner"
    )
)

market_transition.head()

,destination,cluster_2023,cluster_2024_pred
0,Afghanistan,0,1
1,Albania,1,1
2,Algeria,0,1
3,Andorra,5,5
4,Angola,0,1


In [21]:
transition_matrix = pd.crosstab(
    market_transition["cluster_2023"],
    market_transition["cluster_2024_pred"],
    rownames=["2023 Cluster"],
    colnames=["2024 Cluster（使用2023旧模型）"]
)

transition_matrix

2024 Cluster（使用2023旧模型）,0,1,2,3,4,5
2023 Cluster,,,,,,
0,15,24,0,1,5,0
1,18,25,0,0,7,1
2,0,8,2,0,0,10
3,2,0,2,2,0,4
4,1,5,0,0,53,0
5,3,1,0,3,0,2


In [22]:
market_transition["changed"] = (
    market_transition["cluster_2023"]
    != market_transition["cluster_2024_pred"]
)

print(
    "Markets compared:",
    len(market_transition)
)

print(
    "Stable markets:",
    (~market_transition["changed"]).sum()
)

print(
    "Changed markets:",
    market_transition["changed"].sum()
)

print(
    "Transition rate:",
    round(
        market_transition["changed"].mean(),
        4
    )
)

Markets compared: 194
Stable markets: 99
Changed markets: 95
Transition rate: 0.4897


In [23]:
cluster_stability = (
    market_transition
    .groupby("cluster_2023")
    .agg(
        market_count=("destination", "count"),
        stable_markets=("changed", lambda x: (~x).sum()),
        changed_markets=("changed", "sum"),
        transition_rate=("changed", "mean")
    )
)

cluster_stability["stability_rate"] = (
    1 - cluster_stability["transition_rate"]
)

cluster_stability.round(4)

,market_count,stable_markets,changed_markets,transition_rate,stability_rate
cluster_2023,,,,,
0,45,15,30,0.6667,0.3333
1,51,25,26,0.5098,0.4902
2,20,2,18,0.9000,0.1000
3,10,2,8,0.8000,0.2000
4,59,53,6,0.1017,0.8983
5,9,2,7,0.7778,0.2222


In [24]:
historical_cluster_labels = {
    0: "成熟核心市场",
    1: "低基数暴增市场",
    2: "低活跃回落市场",
    3: "高增长机会市场",
    4: "边缘脉冲市场",
    5: "常规培育市场"
}

features_2023["market_segment_2023"] = (
    features_2023["cluster_2023"]
    .map(historical_cluster_labels)
)

features_2024["market_segment_2024_pred"] = (
    features_2024["cluster_2024_pred"]
    .map(historical_cluster_labels)
)

In [25]:
def percentile_score(series, higher_is_better=True):
    rank = series.rank(method="average", ascending=True)
    n = rank.notna().sum()

    score = (rank - 1) / (n - 1) * 100

    if not higher_is_better:
        score = 100 - score

    return score

In [26]:
 def add_opportunity_score(df):
    result = df.copy()

    result["size_score"] = percentile_score(
        result["export_target"]
    )

    result["growth_score"] = percentile_score(
        result["growth_symmetric"]
    )

    result["momentum_score"] = percentile_score(
        result["momentum_symmetric"]
    )

    result["stability_score"] = percentile_score(
        result["monthly_cv"],
        higher_is_better=False
    )

    result["activity_score"] = percentile_score(
        result["active_month_ratio"]
    )

    result["opportunity_score"] = (
        result["size_score"] * 0.30
        + result["growth_score"] * 0.25
        + result["momentum_score"] * 0.20
        + result["stability_score"] * 0.15
        + result["activity_score"] * 0.10
    )

    result["opportunity_rank"] = (
        result["opportunity_score"]
        .rank(
            method="min",
            ascending=False
        )
        .astype(int)
    )

    return result

In [27]:
features_2023 = add_opportunity_score(features_2023)
features_2024 = add_opportunity_score(features_2024)

In [28]:
opportunity_change = (
    features_2023[
        [
            "destination",
            "market_segment_2023",
            "opportunity_score",
            "opportunity_rank"
        ]
    ]
    .rename(
        columns={
            "opportunity_score": "score_2023",
            "opportunity_rank": "rank_2023"
        }
    )
    .merge(
        features_2024[
            [
                "destination",
                "market_segment_2024_pred",
                "opportunity_score",
                "opportunity_rank",
                "export_target",
                "growth_symmetric",
                "momentum_symmetric"
            ]
        ].rename(
            columns={
                "opportunity_score": "score_2024",
                "opportunity_rank": "rank_2024",
                "export_target": "export_2024",
                "growth_symmetric": "growth_2024",
                "momentum_symmetric": "momentum_2024"
            }
        ),
        on="destination",
        how="inner"
    )
)

opportunity_change["score_change"] = (
    opportunity_change["score_2024"]
    - opportunity_change["score_2023"]
)

# 正数 = 排名上升
opportunity_change["rank_improvement"] = (
    opportunity_change["rank_2023"]
    - opportunity_change["rank_2024"]
)

In [29]:
biggest_risers = (
    opportunity_change
    .sort_values(
        "rank_improvement",
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

biggest_risers

,destination,market_segment_2023,score_2023,rank_2023,market_segment_2024_pred,score_2024,rank_2024,export_2024,growth_2024,momentum_2024,score_change,rank_improvement
0,Sierra Leone,低基数暴增市场,22.717949,186,成熟核心市场,64.719388,45,8251574.0,1.359083,0.455659,42.001439,141
1,Ghana,低基数暴增市场,34.025641,160,成熟核心市场,67.321429,28,28610094.0,0.950404,0.925504,33.295788,132
2,Zambia,低基数暴增市场,36.871795,151,成熟核心市场,70.510204,20,34999502.0,1.558654,1.570167,33.638409,131
3,Romania,低基数暴增市场,43.179487,129,边缘脉冲市场,77.627551,4,142004192.0,1.316463,0.439807,34.448064,125
4,Serbia,低基数暴增市场,41.769231,133,成熟核心市场,73.622449,10,36997999.0,1.818780,1.590582,31.853218,123
5,Côte d'Ivoire,低基数暴增市场,38.564103,147,成熟核心市场,65.714286,36,19963323.0,0.855690,0.798396,27.150183,111
6,Kyrgyzstan,低基数暴增市场,28.076923,177,成熟核心市场,60.204082,68,4311378.0,0.307573,1.256751,32.127159,109
7,Curaçao,低基数暴增市场,39.641026,146,成熟核心市场,64.668367,46,6846330.0,1.622829,1.837835,25.027342,100
8,Latvia,低基数暴增市场,48.871795,112,成熟核心市场,73.086735,13,46507639.0,1.892111,1.627600,24.214940,99
9,Oman,低基数暴增市场,43.000000,130,成熟核心市场,66.683673,31,8889822.0,0.592931,1.205279,23.683673,99


In [30]:
biggest_risers[
    [
        "destination",
        "market_segment_2023",
        "market_segment_2024_pred",
        "rank_2023",
        "rank_2024",
        "rank_improvement"
    ]
].head(20)

,destination,market_segment_2023,market_segment_2024_pred,rank_2023,rank_2024,rank_improvement
0,Sierra Leone,低基数暴增市场,成熟核心市场,186,45,141
1,Ghana,低基数暴增市场,成熟核心市场,160,28,132
2,Zambia,低基数暴增市场,成熟核心市场,151,20,131
3,Romania,低基数暴增市场,边缘脉冲市场,129,4,125
4,Serbia,低基数暴增市场,成熟核心市场,133,10,123
5,Côte d'Ivoire,低基数暴增市场,成熟核心市场,147,36,111
6,Kyrgyzstan,低基数暴增市场,成熟核心市场,177,68,109
7,Curaçao,低基数暴增市场,成熟核心市场,146,46,100
8,Latvia,低基数暴增市场,成熟核心市场,112,13,99
9,Oman,低基数暴增市场,成熟核心市场,130,31,99


In [31]:
focus_markets = [
    "Sierra Leone",
    "Ghana",
    "Romania",
    "Serbia",
    "Zambia"
]

In [32]:
compare_2023 = (
    features_2023[
        features_2023["destination"].isin(focus_markets)
    ][
        [
            "destination",
            "export_target",
            "growth_symmetric",
            "momentum_symmetric",
            "monthly_cv",
            "active_month_ratio",
            "opportunity_score",
            "opportunity_rank"
        ]
    ]
    .rename(
        columns={
            "export_target": "export_2023",
            "growth_symmetric": "growth_2023",
            "momentum_symmetric": "momentum_2023",
            "monthly_cv": "cv_2023",
            "active_month_ratio": "activity_2023",
            "opportunity_score": "score_2023",
            "opportunity_rank": "rank_2023"
        }
    )
)

compare_2024 = (
    features_2024[
        features_2024["destination"].isin(focus_markets)
    ][
        [
            "destination",
            "export_target",
            "growth_symmetric",
            "momentum_symmetric",
            "monthly_cv",
            "active_month_ratio",
            "opportunity_score",
            "opportunity_rank"
        ]
    ]
    .rename(
        columns={
            "export_target": "export_2024",
            "growth_symmetric": "growth_2024",
            "momentum_symmetric": "momentum_2024",
            "monthly_cv": "cv_2024",
            "active_month_ratio": "activity_2024",
            "opportunity_score": "score_2024",
            "opportunity_rank": "rank_2024"
        }
    )
)

focus_comparison = compare_2023.merge(
    compare_2024,
    on="destination",
    how="inner"
)

# 为了业务解释，再补一个普通出口额同比
focus_comparison["export_growth_pct"] = (
    (
        focus_comparison["export_2024"]
        / focus_comparison["export_2023"]
    ) - 1
) * 100

focus_comparison[
    [
        "destination",
        "export_2023",
        "export_2024",
        "export_growth_pct",
        "growth_2023",
        "growth_2024",
        "momentum_2023",
        "momentum_2024",
        "cv_2023",
        "cv_2024",
        "rank_2023",
        "rank_2024"
    ]
].round(3)

,destination,export_2023,export_2024,export_growth_pct,growth_2023,growth_2024,momentum_2023,momentum_2024,cv_2023,cv_2024,rank_2023,rank_2024
0,Ghana,10177947.0,28610094.0,181.099,-0.058,0.950,-1.133,0.926,1.975,1.593,160,28
1,Romania,29267639.0,142004192.0,385.192,-0.548,1.316,-0.554,0.440,0.628,0.424,129,4
2,Serbia,1755741.0,36997999.0,2007.258,-0.335,1.819,0.282,1.591,0.491,1.377,133,10
3,Sierra Leone,1574410.0,8251574.0,424.106,-0.218,1.359,-0.543,0.456,1.388,1.058,186,45
4,Zambia,4340651.0,34999502.0,706.319,0.074,1.559,-0.237,1.570,1.219,1.154,151,20


In [33]:
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# 全市场年度机会变化
opportunity_change.to_csv(
    output_dir / "market_opportunity_change_2023_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

# Top 机会跃升市场
biggest_risers.to_csv(
    output_dir / "top_market_opportunity_risers_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Backtest results exported.")

Backtest results exported.


## Backtest Conclusion

本 Notebook 使用 2022–2023 年数据重新构建历史市场识别模型，
并冻结 2023 年的 StandardScaler 与 KMeans 聚类中心，
将 2024 年数据作为模型未见过的新一期数据进行时间外应用。

### 1. 跨年度结构稳定性

2023 历史模型对 2024 市场的分类结果，与直接使用 2024 数据重新训练得到的市场结构相比：

- Adjusted Rand Index (ARI) = 0.7476

说明全球出口市场的整体分群结构具有较好的跨年度延续性。

### 2. 市场类型迁移

成熟核心市场表现出较高稳定性，
其中历史 Cluster 0 的年度稳定率约为 94.9%。

相比之下，高增长、培育及低基数市场具有更明显的年度迁移，
说明机会市场本身具有较强动态性，需要持续更新数据进行监测。

### 3. 新兴机会识别

2023–2024 年机会排名明显提升的市场包括：

- Sierra Leone
- Ghana
- Romania
- Serbia
- Zambia

进一步检查真实贸易指标发现，
这些排名变化普遍伴随着出口规模增长、年度增长改善或近期动量增强。

其中 Romania 同时表现出规模扩大、增长增强和波动下降，
属于较高质量的市场升级案例；

Serbia、Ghana、Zambia 等表现出明显的新兴增长信号，
但仍需要结合绝对规模和波动风险进一步验证。

### 4. 模型定位

本项目的 KMeans 模型应定位为：

**动态市场分层与机会变化监测工具**

而不是预测下一年度出口额或提前预测未来市场机会的监督学习模型。

当新一期贸易数据到来后，
系统可以使用已经冻结的 Scaler 与 KMeans 模型进行 transform + predict，
快速识别市场类型变化，并结合 Opportunity Score 更新市场优先级。

若未来需要预测“下一年度哪些市场将成为高价值机会市场”，
应进一步构建具有明确 t → t+1 预测目标的监督学习模块。